# Test pingouin.rm_corr on built-in 'rm_corr' dataset
This notebook runs the user's snippet, performs programmatic checks, visualizes per-subject data, and provides pytest tests to validate behavior (including DataFrame.copy() cases).

In [1]:
# Install & imports (use local src)
import sys, os, platform
sys.path.insert(0, os.path.abspath("/workspaces/pingouin/src"))
import pingouin as pg
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pytest

print('pingouin:', pg.__version__)
print('pandas :', pd.__version__)
print('numpy  :', np.__version__)
print('python :', platform.python_version())
%matplotlib inline

pingouin: 0.5.5
pandas : 3.0.0
numpy  : 2.4.2
python : 3.12.1


# Load 'rm_corr' dataset and inspect
df = pg.read_dataset('rm_corr')
print('shape:', df.shape)
df.head()

In [11]:
# Sanity checks
assert all(c in df.columns for c in ['pH', 'PacO2', 'Subject']), 'missing expected columns'
print('n_subjects:', df['Subject'].nunique())
print('missing values (pH, PacO2):', df[['pH','PacO2']].isna().sum().to_dict())
print(df[['pH','PacO2']].describe())

n_subjects: 8
missing values (pH, PacO2): {'pH': 0, 'PacO2': 0}
              pH      PacO2
count  47.000000  47.000000
mean    7.172553   4.903404
std     0.268405   0.743287
min     6.330000   3.230000
25%     7.150000   4.375000
50%     7.290000   4.930000
75%     7.335000   5.325000
max     7.420000   6.850000


In [4]:
# Run the user's snippet (original and copy)
df = pg.read_dataset('rm_corr')
print('running rm_corr on original df...')
try:
    res = pg.rm_corr(data=df, x='pH', y='PacO2', subject='Subject')
    print('OK — result:')
    print(res)
except Exception as exc:
    import traceback
    traceback.print_exc()

print('\nrunning rm_corr on df.copy()...')
try:
    res_copy = pg.rm_corr(data=df.copy(), x='pH', y='PacO2', subject='Subject')
    print('OK — copy result:')
    print(res_copy)
except Exception as exc:
    import traceback
    traceback.print_exc()

running rm_corr on original df...
OK — result:
               r  dof      pval            CI95     power
rm_corr -0.50677   38  0.000847  [-0.71, -0.23]  0.929579

running rm_corr on df.copy()...
OK — copy result:
               r  dof      pval            CI95     power
rm_corr -0.50677   38  0.000847  [-0.71, -0.23]  0.929579


In [5]:
# Programmatic validation
import pandas as pd
assert isinstance(res, pd.DataFrame), 'rm_corr must return a DataFrame'
assert 'r' in res.columns and 'pval' in res.columns
r = float(res.at['rm_corr','r'])
pval = float(res.at['rm_corr','pval'])
print('r =', r, 'pval =', pval)
assert -1 <= r <= 1
assert 0 <= pval <= 1

r = -0.5067697422330649 pval = 0.0008471081091288756


In [7]:
# Synthetic reproducible example (should give high within-subject r)
np.random.seed(0)
rows = []
for subj in range(12):
    x = np.linspace(0, 1, 8)
    y = 0.8 * x + np.random.normal(scale=0.03, size=x.size)
    for xi, yi in zip(x, y):
        rows.append({'Subject': subj, 'x': xi, 'y': yi})

df_syn = pd.DataFrame(rows)
res_syn = pg.rm_corr(data=df_syn, x='x', y='y', subject='Subject')
print(res_syn)
r_syn = float(res_syn.at['rm_corr', 'r'])
assert r_syn > 0.7, 'synthetic example should have high within-subject correlation'

               r  dof          pval         CI95  power
rm_corr  0.99474   83  6.432805e-84  [0.99, 1.0]    1.0
